# Aim 1 — COMPASS on Parse 10M PBMC

**Question:** where in the structure of the cytokine response do individuals differ?
- **u**: shared response axis (overall direction of the response)
- **beta**: position of each cytokine along the axis
- **r**: cytokine-specific residual

**Steps**
0. Setup and a quick COMPASS sanity check
1. Configuration (check column names here)
2. Read data in chunks and build pseudobulks
3. Build response vectors per donor x cell type x cytokine
4. Decompose each donor
5. Compare donors
6. Leave-one-donor-out with component swapping (main result)
7. Interferon donor group vs the rest
8. Cytokine-family holdout
9. How many donors are needed?
10. Save results

In [3]:
import os, shutil, subprocess

DATA_DIR = "data/parse10m"
os.makedirs(DATA_DIR, exist_ok=True)

FILES = {
    "h5ad": "https://parse-wget.s3.us-west-2.amazonaws.com/10m/10M_PBMC_12donor_90cytokines_h5ad_20260203.zip",
    "pseudobulks": "https://parse-wget.s3.us-west-2.amazonaws.com/10m/pseudobulks_20260203.zip",  # optional
}

# Check free disk space (zip + extracted needs ~100 GB)
free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
print(f"Free disk: {free_gb:.0f} GB")

def download(url, out_dir):
    out = os.path.join(out_dir, os.path.basename(url))
    # -c resumes an interrupted download
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", out, url], check=True)
    return out

def unzip(path, out_dir):
    subprocess.run(["unzip", "-n", "-q", path, "-d", out_dir], check=True)

for name, url in FILES.items():
    print(f"Downloading {name} ...")
    z = download(url, DATA_DIR)
    print(f"Extracting {name} ...")
    unzip(z, DATA_DIR)
    # os.remove(z)  # uncomment to delete the zip after extracting and save space

print("\nFiles:")
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1e9:6.1f} GB  {p}")

Free disk: 59325 GB
Extracting h5ad ...
Extracting pseudobulks ...

Files:
   0.0 GB  data/parse10m/README
   0.0 GB  data/parse10m/LICENSE
  44.4 GB  data/parse10m/20260203_Parse_10M_PBMC_cytokines.h5ad
   6.2 GB  data/parse10m/pseudobulks_20260203.zip
  44.4 GB  data/parse10m/10M_PBMC_12donor_90cytokines_h5ad_20260203.zip
   0.0 GB  data/parse10m/pseudobulks/pbulk_MAIT_IL-17F.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_Granulocyte_IGF-1.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_CD4_Naive_T_cell__IL-26.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_CD14_Mono_ADSF.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_Plasmablast_IL-3.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_Mono_GM-CSF.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_NK_CD56hi_M-CSF.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_Granulocyte_LT-alpha2-beta1.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_MAIT_LT-alpha2-beta1.h5ad
   0.0 GB  data/parse10m/pseudobulks/pbulk_pDC_IL-2.h5ad
   0.0 GB  data/parse1

In [4]:
import os, subprocess, h5py
import anndata as ad

DATA_DIR = "data/parse10m"
URLS = {
    "10M_PBMC_12donor_90cytokines_h5ad_20260203.zip":
        "https://parse-wget.s3.us-west-2.amazonaws.com/10m/10M_PBMC_12donor_90cytokines_h5ad_20260203.zip",
    "pseudobulks_20260203.zip":
        "https://parse-wget.s3.us-west-2.amazonaws.com/10m/pseudobulks_20260203.zip",
}

# 1) Local size must match the size reported by the server
print("1) Size check")
for fname, url in URLS.items():
    local = os.path.getsize(os.path.join(DATA_DIR, fname))
    head = subprocess.run(["curl", "-sI", url], capture_output=True, text=True).stdout
    remote = next((int(l.split(":")[1]) for l in head.splitlines()
                   if l.lower().startswith("content-length")), None)
    status = "OK" if remote == local else "MISMATCH"
    print(f"   {fname}: local={local:,}  remote={remote:,}  -> {status}")

# 2) Zip integrity (CRC check of every entry; takes a few minutes)
print("\n2) Zip integrity")
for fname in URLS:
    r = subprocess.run(["unzip", "-tq", os.path.join(DATA_DIR, fname)],
                       capture_output=True, text=True)
    print(f"   {fname}: {r.stdout.strip() or r.stderr.strip()}")

# 3) The h5ad opens and has the expected shape (backed mode, no RAM load)
print("\n3) h5ad check")
path = os.path.join(DATA_DIR, "20260203_Parse_10M_PBMC_cytokines.h5ad")
adata = ad.read_h5ad(path, backed="r")
print(f"   shape: {adata.n_obs:,} cells x {adata.n_vars:,} genes  (expected ~9.7M cells)")
print(f"   obs columns: {adata.obs.columns.tolist()}")
with h5py.File(path, "r") as f:
    print(f"   top-level keys: {list(f.keys())}")
    if "layers" in f:
        print(f"   layers: {list(f['layers'].keys())}")

# Also show the README shipped with the data
print("\nREADME:\n" + open(os.path.join(DATA_DIR, "README")).read())

1) Size check
   10M_PBMC_12donor_90cytokines_h5ad_20260203.zip: local=44,367,603,521  remote=44,367,603,521  -> OK
   pseudobulks_20260203.zip: local=6,219,160,912  remote=6,219,160,912  -> OK

2) Zip integrity
   10M_PBMC_12donor_90cytokines_h5ad_20260203.zip: No errors detected in compressed data of data/parse10m/10M_PBMC_12donor_90cytokines_h5ad_20260203.zip.
   pseudobulks_20260203.zip: No errors detected in compressed data of data/parse10m/pseudobulks_20260203.zip.

3) h5ad check
   shape: 9,697,974 cells x 40,352 genes  (expected ~9.7M cells)
   obs columns: ['sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type']
   top-level keys: ['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']
   layers: []

README:
10M PBMC 12donor 90cyto

In [5]:
import anndata as ad
import numpy as np

path = "data/parse10m/20260203_Parse_10M_PBMC_cytokines.h5ad"
adata = ad.read_h5ad(path, backed="r")
obs = adata.obs

print("DONORS:")
print(obs["donor"].value_counts().sort_index().to_string(), "\n")

print("CELL TYPES:")
print(obs["cell_type"].value_counts().to_string(), "\n")

cyt = obs["cytokine"].value_counts()
print(f"CYTOKINES: {len(cyt)} values")
print(sorted(cyt.index.tolist()), "\n")
print("Control-like labels:", [c for c in cyt.index if "PBS" in str(c).upper()], "\n")

print("TREATMENT column (first values):")
print(obs["treatment"].value_counts().head(10).to_string(), "\n")

# Confirm X holds raw integer counts
x = adata.X[:1000]
x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
print("X integer-valued:", np.allclose(x, np.round(x)), "| max:", x.max())

DONORS:
donor
Donor1     1267690
Donor2      931569
Donor3      695158
Donor4      578242
Donor5     1066725
Donor6      974030
Donor7      652386
Donor8      650974
Donor9      632341
Donor10     580786
Donor11     613365
Donor12    1054708 

CELL TYPES:
cell_type
CD4 Memory               2376445
CD4 Naive                1725983
CD14 Mono                1582568
CD8 Memory                757585
B Naive                   625239
CD8 Naive                 619816
NK                        512565
B Intermediate/Memory     326661
MAIT                      322923
CD16 Mono                 232755
Treg                      168248
NKT                       158017
NK CD56bright             125729
cDC                       112944
pDC                        18942
HSPC                       17322
ILC                         8356
Plasmablast                 5876 

CYTOKINES: 91 values
['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'C5a', 'CD27L', 'CD30L', 'CD40L', 'CT-1', 'Decorin', 'EGF', 'EPO', 'FGF-beta', 'F